# Lecture 10: Annotated Text Corpora II

Last time we looked at what linguistic annotation is, explored POS tagsets,
and worked with constituency trees and the Penn Treebank.

Today we'll cover:
- **Dependency grammar** — a different way to represent syntactic structure
- **Universal Dependencies** — the dominant modern annotation framework
- **CoNLL-U format** — and how to work with it in Python
- **BIO tagging** — a flexible annotation scheme for many tasks
- A brief survey of other annotation types

**Prerequisites:**
- Python and the `ling250` environment (updated — see setup below)
- Familiarity with constituency trees and bracket notation from Lecture 9

### Setup

This lecture introduces the `conllu` library for parsing Universal
Dependencies data. Update your environment before running this notebook:

```bash
conda env update -f environment.yaml --prune
```

In [ ]:
import conllu
from collections import Counter
from pathlib import Path


def print_conllu_table(sent, columns=None):
    """Print a CoNLL-U sentence as a formatted table.

    Parameters
    ----------
    sent : conllu.models.TokenList
        A parsed CoNLL-U sentence.
    columns : list of str, optional
        Which token fields to display. Defaults to
        ['id', 'form', 'upos', 'head', 'deprel', 'lemma'].
    """
    if columns is None:
        columns = ['id', 'form', 'upos', 'head', 'deprel', 'lemma']

    # Column display widths
    widths = {
        'id': 3, 'form': 15, 'lemma': 12, 'upos': 6,
        'xpos': 6, 'head': 4, 'deprel': 12, 'feats': 30,
    }

    # Header
    header_parts = []
    for col in columns:
        w = widths.get(col, 12)
        label = col.upper()
        if col in ('id', 'head'):
            header_parts.append(f"{label:>{w}}")
        else:
            header_parts.append(f"{label:{w}}")
    print("  ".join(header_parts))
    print("-" * len("  ".join(header_parts)))

    # Rows
    for token in sent:
        parts = []
        for col in columns:
            w = widths.get(col, 12)
            val = token[col]
            if val is None:
                val = "_"
            elif isinstance(val, dict):
                val = "|".join(f"{k}={v}" for k, v in val.items())
            val = str(val)
            if col in ('id', 'head'):
                parts.append(f"{val:>{w}}")
            else:
                parts.append(f"{val:{w}}")
        print("  ".join(parts))

---

## Part 1: Dependency Grammar

Last time, we represented syntactic structure using **constituency trees**:
words group into phrases (NP, VP, PP, ...), and phrases group into
larger phrases. This is sometimes called **phrase structure grammar**.

But there's a different way to think about sentence structure:
instead of asking "which words group together?", we can ask
"which words depend on which other words?"

In **dependency grammar**, every word in a sentence is connected to
exactly one other word (its **head**) by a labeled relationship.
One word — typically the main verb — is the **root** of the whole
sentence and has no head of its own.

Consider: **"The big dog chased the cat"**

As a **constituency tree** (what we saw last time):
```
              S
      ________|________
     NP                VP
  ___|_____        ____|____
 DT  JJ   NN    VBD        NP
 |   |     |     |       ___|___
The big   dog  chased   DT     NN
                        |      |
                       the    cat
```

As a **dependency tree**:
```
        chased
        ___|____
      dog      cat
    __|__       |
  The   big    the
```

In the dependency view:
- "chased" is the **root** (the main verb)
- "dog" depends on "chased" — it's the subject (**nsubj**)
- "cat" depends on "chased" — it's the object (**obj**)
- "The" depends on "dog" — it's a determiner (**det**)
- "big" depends on "dog" — it's an adjectival modifier (**amod**)
- "the" depends on "cat" — it's a determiner (**det**)

### Constituency vs. dependency

These are two different ways to annotate the same underlying structure:

| | Constituency | Dependency |
|---|---|---|
| **Asks** | What groups together? | What depends on what? |
| **Nodes** | Phrases (NP, VP, S) | Words |
| **Labels** | Phrase types | Grammatical relations (nsubj, obj, det) |
| **Classic corpus** | Penn Treebank | Universal Dependencies |

Both capture important information. But dependency grammar has become
the dominant framework in modern computational linguistics, for a few
reasons:

1. **Cross-linguistic applicability**: Phrase structure rules vary a lot
   across languages (is the verb at the end? in the middle?), but
   dependency relations (subject, object, modifier) are more universal
2. **Simpler structure**: Every word connects to exactly one head —
   there are no intermediate phrase nodes
3. **Directly useful**: Dependency relations tell you *who did what to
   whom* — exactly what you need for many downstream tasks

### Common dependency relations

Some of the most important relations you'll see:

| Relation | Meaning | Example |
|----------|---------|----------|
| **nsubj** | nominal subject | *The **dog** chased the cat* → dog is nsubj of chased |
| **obj** | direct object | *The dog chased the **cat*** → cat is obj of chased |
| **det** | determiner | ***The** dog* → The is det of dog |
| **amod** | adjectival modifier | *The **big** dog* → big is amod of dog |
| **advmod** | adverbial modifier | *ran **quickly*** → quickly is advmod of ran |
| **case** | case marker (preposition) | ***in** the park* → in is case of park |
| **nmod** | nominal modifier | *mosque **in the town*** → town is nmod of mosque |
| **root** | root of the sentence | The main verb (or predicate) |
| **punct** | punctuation | Periods, commas, etc. |

---

## Part 2: Universal Dependencies

The **Universal Dependencies (UD)** project is a massive collaborative
effort to create consistently annotated treebanks across many languages.
As of 2024, UD covers **over 150 treebanks in 100+ languages** — from
English and Chinese to Yoruba and Erzya.

The key idea: a single set of POS tags, morphological features, and
dependency relations that work across all languages. This makes it
possible to directly compare syntactic patterns across languages, or
to build tools that work on many languages at once.

**UD website**: https://universaldependencies.org/

### UPOS tags vs. other tagsets

UD defines its own POS tagset called **UPOS** (Universal POS). It has
17 tags. You've already seen two other tagsets — the Penn Treebank
tagset and the "universal" tagset in NLTK. These are all related but
**not identical**:

| NLTK `tagset='universal'` | UD UPOS | Notes |
|---------------------------|---------|-------|
| NOUN | NOUN | Same |
| VERB | VERB | Same |
| ADJ | ADJ | Same |
| ADV | ADV | Same |
| DET | DET | Same |
| ADP | ADP | Same |
| PRON | PRON | Same |
| CONJ | CCONJ / SCONJ | UD splits coordinating vs. subordinating |
| NUM | NUM | Same |
| PRT | PART | Different name |
| X | X | Same |
| . | PUNCT / SYM | UD distinguishes punctuation from symbols |
| — | AUX | UD has a separate tag for auxiliary verbs |
| — | PROPN | UD distinguishes proper nouns from common nouns |
| — | INTJ | UD has a tag for interjections |

The NLTK "universal" tagset (12 tags) is a simplified predecessor that
inspired UPOS, but UPOS (17 tags) is more refined and is the current
standard.

### The CoNLL-U format

UD treebanks are stored in **CoNLL-U format** — a tab-separated,
plain-text format with one token per line. It's essentially a TSV file
with specific columns and some conventions.

Here's a real example from the English UD treebank:

```
# text = From the AP comes this story :
1	From	from	ADP	IN	_	3	case	3:case	_
2	the	the	DET	DT	Definite=Def|PronType=Art	3	det	3:det	_
3	AP	AP	PROPN	NNP	Number=Sing	4	obl	4:obl:from	_
4	comes	come	VERB	VBZ	Mood=Ind|Number=Sing|...	0	root	0:root	_
5	this	this	DET	DT	Number=Sing|PronType=Dem	6	det	6:det	_
6	story	story	NOUN	NN	Number=Sing	4	nsubj	4:nsubj	_
7	:	:	PUNCT	:	_	4	punct	4:punct	_
```

The 10 columns are:

| # | Column | Description | Example |
|---|--------|-------------|----------|
| 1 | **ID** | Token index (1-based) | 3 |
| 2 | **FORM** | The word as it appears | AP |
| 3 | **LEMMA** | Base form / dictionary form | AP |
| 4 | **UPOS** | Universal POS tag | PROPN |
| 5 | **XPOS** | Language-specific POS tag | NNP |
| 6 | **FEATS** | Morphological features | Number=Sing |
| 7 | **HEAD** | ID of this token's head (0 = root) | 4 |
| 8 | **DEPREL** | Dependency relation to head | obl |
| 9 | **DEPS** | Enhanced dependencies | 4:obl:from |
| 10 | **MISC** | Miscellaneous annotations | _ |

Key conventions:
- `_` means "not specified" (like `None`)
- Blank lines separate sentences
- Lines starting with `#` are comments (often containing the original
  sentence text and a sentence ID)
- HEAD=0 means this word is the **root** of the sentence

### Morphological features

Column 6 (FEATS) contains morphological features as `Key=Value` pairs
separated by `|`. These encode grammatical properties like:

- `Number=Sing` or `Number=Plur`
- `Tense=Past` or `Tense=Pres`
- `Case=Nom` or `Case=Acc` (in languages with case marking)
- `PronType=Art` (article), `PronType=Dem` (demonstrative)
- `Mood=Ind` (indicative), `VerbForm=Fin` (finite)

These features are especially rich in morphologically complex
languages — a single Turkish verb form might have 5+ features. English
is relatively sparse by comparison.

---

## Part 3: Working with CoNLL-U in Python

We'll use the `conllu` library to parse and explore UD data. The
demo file `ud_english_sample.conllu` contains 15 sentences from the
English UD treebank (UD_English-EWT).

In [ ]:
# Load the sample file
sample_path = Path("../demos/annotated_text/ud_english_sample.conllu")

with open(sample_path) as f:
    sentences = conllu.parse(f.read())

print(f"Loaded {len(sentences)} sentences")

### Sentence structure

`conllu.parse()` returns a list of `TokenList` objects — one per sentence.
Each `TokenList` contains the tokens and any metadata (like the original
text).

In [ ]:
sent = sentences[0]

# Metadata comes from the # comment lines
print("Metadata:")
print(f"  text: {sent.metadata['text']}")
print(f"  sent_id: {sent.metadata['sent_id']}")

In [ ]:
# Each token is a dictionary with the 10 CoNLL-U fields
token = sent[0]
print("Token keys:", list(token.keys()))
print()
print("First token:")
for key, value in token.items():
    print(f"  {key:8s} = {value}")

In [ ]:
# Print all tokens in a sentence with key fields
print_conllu_table(sent)

### Reading the dependency structure

The HEAD column tells you which word each token depends on. HEAD=0
means it's the root. Let's trace the structure:

In [ ]:
# For each token, show what it depends on
print(f"Sentence: {sent.metadata['text']}")
print()
for token in sent:
    if token['head'] == 0:
        print(f"  '{token['form']}' is the ROOT")
    else:
        # Find the head token
        head_form = sent[token['head'] - 1]['form']
        print(f"  '{token['form']}' --{token['deprel']}--> '{head_form}'")

In [ ]:
# Let's look at a couple more sentences
for s in sentences[4:6]:
    print(f"Sentence: {s.metadata['text']}")
    print_conllu_table(s, columns=['id', 'form', 'upos', 'head', 'deprel'])
    print()

### Morphological features

The `feats` field is parsed into a dictionary (or `None` if empty).

In [ ]:
# Show tokens with their morphological features
print_conllu_table(sentences[0], columns=['id', 'form', 'upos', 'feats'])

### Searching across sentences

Just like we searched the Penn Treebank for structural patterns last time,
we can search UD data for dependency patterns. Since each token has
clearly labeled relations, this is straightforward.

In [ ]:
# Find all subjects in our sample
print("All nominal subjects (nsubj):")
for sent in sentences:
    for token in sent:
        if token['deprel'] == 'nsubj':
            head_form = sent[token['head'] - 1]['form']
            print(
                f"  {token['form']:15s} --nsubj--> {head_form:15s}"
                f"  ({sent.metadata['text'][:40]}...)"
            )

In [ ]:
# Count dependency relations across the sample
deprel_counts = Counter()
for sent in sentences:
    for token in sent:
        deprel_counts[token['deprel']] += 1

print("Most common dependency relations:")
for rel, count in deprel_counts.most_common(12):
    print(f"  {rel:15s} {count:4d}")

In [ ]:
# Count UPOS tags
upos_counts = Counter()
for sent in sentences:
    for token in sent:
        upos_counts[token['upos']] += 1

print("POS tag distribution:")
for tag, count in upos_counts.most_common():
    print(f"  {tag:8s} {count:4d}")

### Serializing back to CoNLL-U

The `conllu` library can also write data back to CoNLL-U format. This
is useful if you modify annotations and want to save them.

In [ ]:
# .serialize() converts a TokenList back to CoNLL-U text
print(sentences[0].serialize())

---

## Part 4: BIO Tagging

So far, all our annotation schemes have labeled individual words (POS
tags) or relationships between words (dependencies). But what if we
want to label **spans** — contiguous sequences of words that form a
meaningful unit?

For example, **Named Entity Recognition (NER)** identifies mentions
of people, places, and organizations in text:

> **Barack Obama** visited **New York City** on **Tuesday**.

The challenge: "New York City" is *three words* but *one entity*.
How do we represent this as a per-word annotation?

### The BIO scheme

The **BIO** (or **IOB**) tagging scheme solves this by using three
tag prefixes:

- **B**-X = **Beginning** of an entity of type X
- **I**-X = **Inside** (continuing) an entity of type X
- **O** = **Outside** any entity

Here's how our example would be tagged:

| Word | BIO Tag |
|------|---------|
| Barack | B-PER |
| Obama | I-PER |
| visited | O |
| New | B-LOC |
| York | I-LOC |
| City | I-LOC |
| on | O |
| Tuesday | B-DATE |
| . | O |

The B- tag marks the **start** of a new entity. The I- tag marks
words that **continue** the same entity. O marks words that aren't
part of any entity.

Why do we need B- at all? Consider two entities next to each other:
"**New York** **Times**" — without B- to mark the start of "Times",
we couldn't tell where one entity ends and the next begins.

In [ ]:
# A BIO-tagged sentence as data
bio_example = [
    ("Barack",  "B-PER"),
    ("Obama",   "I-PER"),
    ("visited", "O"),
    ("New",     "B-LOC"),
    ("York",    "I-LOC"),
    ("City",    "I-LOC"),
    ("on",      "O"),
    ("Tuesday", "B-DATE"),
    (".",       "O"),
]

for word, tag in bio_example:
    print(f"  {word:10s}  {tag}")

In [ ]:
# Extracting entities from BIO tags
def extract_entities(tagged_words):
    """Extract named entities from a BIO-tagged word list."""
    entities = []
    current_entity = []
    current_type = None

    for word, tag in tagged_words:
        if tag.startswith('B-'):
            # Save previous entity if there was one
            if current_entity:
                entities.append((' '.join(current_entity), current_type))
            # Start a new entity
            current_type = tag[2:]
            current_entity = [word]
        elif tag.startswith('I-') and current_entity:
            # Continue the current entity
            current_entity.append(word)
        else:
            # O tag: save any ongoing entity and reset
            if current_entity:
                entities.append((' '.join(current_entity), current_type))
                current_entity = []
                current_type = None

    # Don't forget the last entity
    if current_entity:
        entities.append((' '.join(current_entity), current_type))

    return entities


entities = extract_entities(bio_example)
print("Extracted entities:")
for text, entity_type in entities:
    print(f"  {text:20s}  [{entity_type}]")

### The power of BIO

BIO tagging was originally designed for NER, but the same scheme can
represent **any** task that involves labeling spans of text. This is
what makes it so powerful — many different problems can be framed as
"assign a BIO tag to each word":

**Noun phrase chunking:**

| The | big | dog | chased | the | cat |
|-----|-----|-----|--------|-----|-----|
| B-NP | I-NP | I-NP | O | B-NP | I-NP |

**Quotation detection:**

| She | said | " | that's | interesting | " | and | left |
|-----|------|---|--------|-------------|---|-----|------|
| O | O | B-QUOTE | I-QUOTE | I-QUOTE | I-QUOTE | O | O |

**Slot filling** (for dialogue systems):

| Book | a | flight | to | New | York | on | Friday |
|------|---|--------|-----|-----|------|-----|--------|
| O | O | O | O | B-DEST | I-DEST | O | B-DATE |

The general pattern: whenever you need to identify *spans* in text
and categorize them, BIO tagging gives you a way to turn it into a
per-word labeling task. This is a powerful framing because many
standard machine learning tools work well on per-word classification.

### BIO in CoNLL format

BIO-tagged data is often stored in a column format similar to CoNLL-U.
Here's what a typical NER dataset looks like:

```
Barack   B-PER
Obama    I-PER
visited  O
New      B-LOC
York     I-LOC
City     I-LOC
on       O
Tuesday  B-DATE
.        O

The      O
meeting  O
...
```

Blank lines separate sentences, just like in CoNLL-U. Some datasets
include additional columns (POS tags, chunk tags, etc.).

---

## Part 5: The Broader Annotation Landscape

POS tags, constituency trees, dependency trees, and BIO tags are the
annotation types you're most likely to encounter. But there are many
other kinds of linguistic annotation. Here's a brief survey, so you'll
recognize them if you come across them:

### Semantic role labeling

**What:** Annotates *who did what to whom, where, when, and how*.

> \[The dog\]<sub>AGENT</sub> **chased** \[the cat\]<sub>PATIENT</sub> \[through the park\]<sub>PATH</sub>

Related to dependency relations (nsubj, obj) but at a more semantic
level.

### Coreference

**What:** Links mentions that refer to the same entity.

> **The dog** chased the cat. **It** ran quickly. **The animal** was tired.

"The dog", "It", and "The animal" all refer to the same entity.
Coreference annotation groups these mentions into clusters.

### Sentiment and stance

**What:** Labels text with attitudes, opinions, or emotional valence.

Can be document-level ("this review is positive"), sentence-level,
or aspect-level ("the **food** was great but the **service** was slow").

### Discourse structure

**What:** Annotates relationships between sentences or clauses — e.g.,
contrast, elaboration, cause-effect.

> \[The dog ran quickly\]<sub>CAUSE</sub> \[because it saw a squirrel\]<sub>EFFECT</sub>

Frameworks include Rhetorical Structure Theory (RST) and the Penn
Discourse Treebank (PDTB).

### Where to find annotated data

- **Universal Dependencies**: https://universaldependencies.org/
  — 150+ treebanks, free and open
- **Linguistic Data Consortium (LDC)**: https://www.ldc.upenn.edu/
  — Penn Treebank, OntoNotes, and many others (many require a license;
  UR has institutional access)
- **HuggingFace Datasets**: https://huggingface.co/datasets
  — Large collection of NLP datasets, many freely downloadable

### Annotation tools

If you ever need to create your own annotations (e.g., for a term
project), some tools to know about:

- **brat** (https://brat.nlplab.org/) — web-based, widely used for NER
  and relation annotation
- **INCEpTION** (https://inception-project.github.io/) — more full-featured,
  supports many annotation types
- **Label Studio** (https://labelstud.io/) — general-purpose annotation
  tool (text, images, audio)

---

## Exercises

Try these on your own!

### Exercise 1: Find all objects

Search the sample CoNLL-U data for all direct objects (`deprel == 'obj'`).
For each one, print the object word and the verb it depends on.

**Hint:** The head of an `obj` token is usually a verb. Use
`sent[token['head'] - 1]` to look up the head.

In [ ]:
# Your code here


### Exercise 2: UPOS vs. XPOS

The CoNLL-U format stores both UPOS (universal) and XPOS (language-specific)
tags for each token. For all verbs (`upos == 'VERB'`) in the sample,
print both tags side by side. How many distinct XPOS tags map to the
single UPOS tag `VERB`?

In [ ]:
# Your code here


### Exercise 3: BIO tagging practice

Write BIO tags for the following sentence, marking person names (PER),
locations (LOC), and organizations (ORG):

> Marie Curie worked at the University of Paris in France.

Then create it as a Python list of `(word, tag)` tuples and use the
`extract_entities()` function from Part 4 to verify your tagging.

In [ ]:
# Your code here


### Exercise 4: Dependency relation patterns

For each root verb in the sample, find whether it has both a subject
(`nsubj` or `nsubj:pass`) and an object (`obj`). Print the
subject-verb-object triple for sentences that have both.

**Hint:** First find the root (HEAD=0), then look for tokens whose
HEAD points to the root's ID.

In [ ]:
# Your code here


---

## Quick Reference

### conllu library

| Code | What it does |
|------|-------------|
| `conllu.parse(text)` | Parse CoNLL-U text into a list of TokenLists |
| `sent.metadata` | Dict of comment-line metadata (text, sent_id, ...) |
| `sent[i]` | Access the i-th token (0-indexed) |
| `token['form']` | Word as it appears |
| `token['lemma']` | Base form |
| `token['upos']` | Universal POS tag |
| `token['xpos']` | Language-specific POS tag |
| `token['feats']` | Morphological features (dict or None) |
| `token['head']` | ID of head token (0 = root) |
| `token['deprel']` | Dependency relation |
| `sent.serialize()` | Convert back to CoNLL-U text |

### print_conllu_table (defined in this notebook)

| Code | What it does |
|------|-------------|
| `print_conllu_table(sent)` | Print sentence with default columns (id, form, upos, head, deprel, lemma) |
| `print_conllu_table(sent, columns=[...])` | Print with specific columns (any token field names) |

### Common UPOS tags

| Tag | Meaning | Tag | Meaning |
|-----|---------|-----|--------|
| NOUN | Common noun | PRON | Pronoun |
| PROPN | Proper noun | DET | Determiner |
| VERB | Verb | ADP | Adposition |
| AUX | Auxiliary verb | CCONJ | Coordinating conj. |
| ADJ | Adjective | SCONJ | Subordinating conj. |
| ADV | Adverb | PART | Particle |
| NUM | Numeral | INTJ | Interjection |
| PUNCT | Punctuation | SYM | Symbol |
| X | Other | | |

### Common dependency relations

| Relation | Meaning |
|----------|--------|
| nsubj | Nominal subject |
| obj | Direct object |
| iobj | Indirect object |
| det | Determiner |
| amod | Adjectival modifier |
| advmod | Adverbial modifier |
| nmod | Nominal modifier |
| case | Case marker (preposition) |
| conj | Conjunct |
| cc | Coordinating conjunction |
| root | Root of sentence |

### BIO tagging

| Prefix | Meaning |
|--------|---------|
| B-X | Beginning of span of type X |
| I-X | Inside (continuation of) span of type X |
| O | Outside any span |